## Automated Reasoning-Chain Annotation via Claude API

Uses an LLM API to label how the model's generated reasoning reacts to an injected counterfactual number (e.g. correcting it, following through, or recovering).

In [18]:
import anthropic
import pandas as pd
import os
import dotenv
dotenv.load_dotenv()

True

### Set-up

In [ ]:
import sys
sys.path.append("../../src")

import _config

prompt_config = _config.PromptConfig(model_type="GPT-OSS_stepwise", prompt_type="h")
manual_annotation_path = _config.build_run_output_path(
    prompt_config,
    _config.RunConfig(
        experiment_root="experiments/token_intervention",
        output_filename="generation_h_filtered_for_manual_annotation.csv",
    ),
)
annotated_manual_annotation_path = manual_annotation_path.with_name(f"annotated_{manual_annotation_path.name}")
eval_annotated_manual_annotation_path = manual_annotation_path.with_name(f"eval_annotated_{manual_annotation_path.name}")
annotated_generation_path = manual_annotation_path.with_name("annotated_generation_h.csv")

### Annotation instruction

Defines the label categories used to classify how the reasoning chain reacts to the injected error; the commented-out cells are earlier drafts of the prompt.

In [2]:
# instruction = '''I want you to annotate a reasoning chain generated by a language model where a researcher has introduced an error to test how the model handles it. Below you will see the reasoning chain up to the error, the natural continuation without an error, the error introduced by the researcher, and the model's continuation provided after the error. I will also provide you the correct final answer, so that you can easily detect the 'Override' cases by checking whether the correct answer occurs before any correction of an error.

# There are four ways in which the model reacts to the error:
# 1. Correction: It corrects it, often involving the word "wait" or a question mark.
# 2. Follow Through: It follows through with the error, propagating it until the final answer, so that the final answer is incorrect.
# 3. Override: The model outputs the correct answer before any explicit correction. There may be a correction right afterwards.
# 4. Recovery: It recovers the correct answer by adding an extra step to the reasoning. This could involve adding or subtracting a necessary quantity to get the reasoning steps back on track.

# Please annotate the reasoning chain with the appropriate label. Output only the label (e.g. "Correction", "Follow Through", "Override", or "Recovery"):
# Prefix: {reasoning_chain}
# Natural continuation: {natural_continuation}
# Error: {error}
# Post-error continuation: {model_continuation}
# Correct final answer: {correct_final_answer}
# '''

In [3]:
# instruction = '''I want you to annotate a reasoning chain generated by a language model where a researcher has introduced an error to test how the model handles it. Below you will see the reasoning chain up to the error, the natural continuation without an error, the error introduced by the researcher, and the model's continuation provided after the error. I will also provide you the correct and incorrect final answers, so that you can easily detect the 'Override' cases by checking whether the correct answer simply overrides the incorrect answer.

# Use the following steps to annotate the reasoning chain:
# 1. Output "Correction" if the Post-error continuation immediately (or very quickly) corrects the error (e.g. "wait" or a question mark).
# 2. Output "Recovery" if the Post-error continuation contains a step (often close to the beginning) that is not found in the Natural continuation. For instance, it might add an extra number or subtract a necessary quantity to get the reasoning steps back on track.
# 3. Output "Follow Through" if the Post-error continuation follows through with the error, and the final answer is incorrect.
# 4. Output "Override" if the Post-error continuation outputs the incorrect answer, followed by "So the sum is", and then followed by the correct answer. It may then correct the error and recalculate the correct answer again. If a reasoning chain is both "Override" and "Recovery," go with "Recovery."

# Please annotate the reasoning chain with the appropriate label. Output only the label (e.g. "Correction", "Follow Through", "Override", or "Recovery"):
# Prefix: {reasoning_chain}
# Natural continuation: {natural_continuation}
# Error: {error}
# Post-error continuation: {model_continuation}
# Correct final answer: {correct_final_answer}
# Incorrect final answer: {incorrect_final_answer}
# '''

In [4]:
# instruction = '''I want you to annotate a reasoning chain generated by a language model where a researcher has introduced an error to test how the model handles it. Below you will see the reasoning chain up to the error, the natural continuation without an error, the error introduced by the researcher, and the model's continuation provided after the error. I will also provide you the correct and incorrect final answers, so that you can easily detect the 'Override' cases by checking whether the correct answer simply overrides the incorrect answer.

# Use the following steps in order to annotate the reasoning chain:
# 1. Output "Follow Through" if the final answer is incorrect.
# 2. Output "Correction" if the Post-error starts off by immediately acknowledging the error (e.g. "wait" or a question mark).
# 3. Output "Override" if the Post-error continuation has this sequence: "= (incorrect_final_answer). So the sum is (correct_final_answer)?" It may still acknowledge the error, but not before the sequence.
# 3. Output "Recovery" if the Post-error continuation gets to the correct answer by adding a step (often close to the beginning) that is not found in the Natural continuation. For instance, it might add an extra number or subtract a necessary quantity to get the reasoning steps back on track. It may still acknowledge the error, but not before the extra step.

# Please annotate the reasoning chain with the appropriate label. Output only the label (e.g. "Correction", "Follow Through", "Override", or "Recovery"):
# Prefix: {reasoning_chain}
# Natural continuation: {natural_continuation}
# Error: {error}
# Post-error continuation: {model_continuation}
# Correct final answer: {correct_final_answer}
# Incorrect final answer: {incorrect_final_answer}
# '''

In [5]:
# instruction = '''I want you to annotate a reasoning chain generated by a language model where a researcher has introduced an error to test how the model handles it. Below you will see the reasoning chain up to the error, the natural continuation without an error, the error introduced by the researcher, and the model's continuation provided after the error. I will also provide you the correct and incorrect final answers, so that you can easily detect the 'Override' cases by checking whether the correct answer simply overrides the incorrect answer.

# Use the following steps in order to annotate the reasoning chain:
# 1. Output "Follow Through" if the final answer is incorrect.
# 2. Output "Correction" if the Post-error starts off by immediately acknowledging the error (e.g. "wait" or a question mark).
# 3. Output "Override" if the Post-error continuation has this sequence: "= {incorrect_final_answer}. So the sum is {correct_final_answer}?" It may still acknowledge the error afterwards, but not before the sequence.
# 4. Output "Recovery" if the Post-error continuation gets to the correct answer by adding a step (often close to the beginning) that is not found in the Natural continuation. For instance, it might add an extra number or subtract a necessary quantity to get the reasoning steps back on track. It may still acknowledge the error, but not before the extra step.

# Please annotate the reasoning chain with the appropriate label. Output only the label (e.g. "Correction", "Follow Through", "Override", or "Recovery"):
# Prefix: {reasoning_chain}
# Natural continuation: {natural_continuation}
# Error: {error}
# Post-error continuation: {model_continuation}
# Correct final answer: {correct_final_answer}
# Incorrect final answer: {incorrect_final_answer}
# '''

In [6]:
instruction = '''I want you to annotate a reasoning chain generated by a language model where a researcher has introduced an error to test how the model handles it. Below you will see the reasoning chain up to the error, the natural continuation without an error, the error introduced by the researcher, and the model's continuation provided after the error. I will also provide you the correct and incorrect final answers, so that you can easily detect the 'Override' cases by checking whether the correct answer simply overrides the incorrect answer.

# Use the following steps in order to annotate the reasoning chain:
# 1. Output "Follow Through" if the final answer is incorrect.
# 2. Output "Correction" if the Post-error starts off by immediately acknowledging the error (e.g. "wait" or a question mark).
# 3. Output "Override" if the Post-error continuation contains both the incorrect final answer and the correct final answer. There must be no acknowledgment of the error before the correct answer is given.
# 4. Output "Recovery" if the Post-error continuation gets to the correct answer by adding a step (often close to the beginning) that is not found in the Natural continuation. For instance, it might add an extra number or subtract a necessary quantity to get the reasoning steps back on track. It may still acknowledge the error, but not before the extra step.

# Please annotate the reasoning chain with the appropriate label. Output only the label (e.g. "Correction", "Follow Through", "Override", or "Recovery"):
# Prefix: {reasoning_chain}
# Natural continuation: {natural_continuation}
# Error: {error}
# Post-error continuation: {model_continuation}
# Correct final answer: {correct_final_answer}
# Incorrect final answer: {incorrect_final_answer}
# '''

### Batch annotation via Claude API

Helper functions for sending rows to the annotation API in bulk and writing back an annotated CSV.

In [7]:
def call_claude_batch_api(client: anthropic.Anthropic, user_prompts: list[str]) -> list[str]:
    """
    Call the Claude API with a batch of user prompts using batch processing.
    
    Args:
        user_prompts (list[str]): List of prompts to send to Claude
        
    Returns:
        list[str]: List of responses from Claude
    """
    
    # Create batch requests
    requests = []
    for i, prompt in enumerate(user_prompts):
        requests.append({
            "custom_id": f"request_{i}",
            "params": {
                "model": "claude-opus-4-6",
                "max_tokens": 2,
                "messages": [
                    {"role": "user", "content": prompt}
                ]
            }
        })
    
    # Create and submit batch
    batch = client.beta.messages.batches.create(requests=requests)
    
    # Wait for batch to complete
    import time
    while batch.processing_status in ["in_progress", "validating"]:
        time.sleep(10)  # Wait 10 seconds before checking again
        batch = client.beta.messages.batches.retrieve(batch.id)
        print(batch.processing_status)
    
    if batch.processing_status == "ended":
        # Retrieve results
        results = client.beta.messages.batches.results(batch.id)
        
        # Sort results by custom_id to maintain order
        sorted_results = sorted(results, key=lambda x: int(x.custom_id.split('_')[1]))
        
        # Extract text responses
        responses = []
        for result in sorted_results:
            if result.result.type == "succeeded":
                responses.append(result.result.message.content[0].text)
            else:
                responses.append(f"Error: {result.result.error}")
        
        return responses
    else:
        raise Exception(f"Batch processing failed with status: {batch.processing_status}")

In [8]:
def annotate_csv_with_api(input_csv_path: str, client: anthropic.Anthropic) -> str:
    """
    Read a CSV file, add annotations using Claude API, and save as a new CSV file.
    
    Args:
        input_csv_path (str): Path to the input CSV file
        
    Returns:
        str: Path to the output CSV file
    """
    # Read the CSV file
    df = pd.read_csv(input_csv_path)
    
    # Create output filename with "annotated_" prefix
    input_dir = os.path.dirname(input_csv_path)
    input_filename = os.path.basename(input_csv_path)
    output_filename = f"annotated_{input_filename}"
    output_path = os.path.join(input_dir, output_filename)
    
    # Collect all formatted instructions for batch processing
    formatted_instructions = []
    
    for index in range(len(df)):
        row = df.iloc[index]
        
        # Format the instruction with the row data
        formatted_instruction = instruction.format(
            reasoning_chain=row['base_before'],
            natural_continuation=str(row['base_number']) + row['base_after'],
            correct_final_answer=str(row['base_sum']),
            incorrect_final_answer=str(row['source_sum']),
            error=str(row['source_number']),
            model_continuation=row['generated_text']
        )
        
        formatted_instructions.append(formatted_instruction)
    
    print(f"Processing {len(formatted_instructions)} rows with batch API")
    
    # Call the batch API to get all annotations
    try:
        annotations = call_claude_batch_api(client, formatted_instructions)
    except Exception as e:
        print(f"Error processing batch: {e}")
        annotations = ["ERROR: Failed to annotate"] * len(formatted_instructions)
    
    # Add annotations to the dataframe
    df['annotation'] = annotations
    
    # Save the annotated dataframe
    df.to_csv(output_path, index=False)
    
    print(f"Annotated CSV saved to: {output_path}")
    return output_path

### Run annotation

In [9]:
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

In [ ]:
annotate_csv_with_api(manual_annotation_path, client)

Processing 60 rows with batch API
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
ended
Annotated CSV saved to: ../../experiments/token_intervention/output/GPT-OSS_stepwise/annotated_generation_h_filtered_for_manual_annotation.csv


'../../experiments/token_intervention/output/GPT-OSS_stepwise/annotated_generation_h_filtered_for_manual_annotation.csv'

### Validate against hand-labeled examples

Checks the auto-annotator's labels against a small hand-labeled sample before trusting it on the full dataset.

In [ ]:
# test_expected_annotations = [
#     "correction", "correction", "correction", "correction", "correction",
#     "correction", "correction", "correction", "correction", "correction",
#     "correction", "correction", "correction", "correction", "recovery",
#     "recovery", "correction", "override", "correction", "double-check",
#     "correction", "override", "correction", "correction", "correction",
#     "correction", "correction", "recovery", "recovery", "override",
#     "correction", "override", "correction", "override", "correction",
#     "correction", "correction", "correction", "correction", "recovery",
#     "correction", "correction", "correction", "follow through", "correction",
#     "follow through", "correction", "correction", "correction", "recovery",
#     "recovery", "recovery", "correction", "override", "correction",
#     "correction", "correction", "correction", "correction", "correction"
# ]


test_expected_annotations = [
        "correction", "correction", "recovery", "recovery", "correction",
        "override", "correction", "correction", "correction", "override",
        "correction", "correction", "correction", "correction", "correction",
        "recovery", "correction", "correction", "correction", "follow through",
        "correction", "follow through", "correction", "correction", "correction",
        "correction", "correction", "correction", "correction", "correction",
        "correction", "correction", "correction", "correction", "correction",
        "correction", "correction", "correction", "correction", "recovery",
        "recovery", "override", "correction", "override", "correction",
        "override", "correction", "correction", "correction", "recovery",
        "recovery", "recovery", "correction", "override", "correction",
        "correction", "correction", "correction", "correction", "correction"
    ]

eval_expected_annotations = [
    "correction",
    "correction",
    "recovery",
    "recovery",
    "correction",
    "override",
    "correction",
    "correction",
    "correction",
    "override",
    "correction",
    "correction",
    "correction",
    "correction",
    "recovery",
    "recovery",
    "recovery",
    "override",
    "correction",
    "correction",
    "correction",
    "override",
    "correction",
    "correction",
    "correction",
    "recovery",
    "recovery",
    "recovery",
    "recovery",
    "override",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "recovery",
    "recovery",
    "recovery",
    "follow through",
    "follow through",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "correction",
    "recovery",
    "recovery",
    "recovery",
    "follow through",
    "correction",
    "correction",
    "correction",
    "follow through",
    "correction",
    "correction"
]

In [ ]:
def score_annotation_file(csv_path, expected_annotations):
    """
    Score an annotated CSV file by comparing the first 60 rows of 'annotation' 
    column against the expected pattern.
    
    Args:
        csv_path (str): Path to the annotated CSV file
        
    Returns:
        dict: Dictionary containing score information
    """
    
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)
        
        # Check if annotation column exists
        if 'annotation' not in df.columns:
            return {
                'error': 'No annotation column found in CSV',
                'score': 0,
                'total': len(expected_annotations),
                'matches': 0
            }
        
        # Get first 60 annotations (or fewer if file has less than 60 rows)
        actual_annotations = df['annotation'].head(60).tolist()
        
        # Compare annotations and track mismatches
        matches = 0
        mismatches = []
        total_compared = min(len(actual_annotations), len(expected_annotations))
        
        for i in range(total_compared):
            actual = str(actual_annotations[i]).strip().lower()
            expected = expected_annotations[i].lower()
            
            if actual == expected:
                matches += 1
            else:
                mismatches.append({
                    'row': i + 1,
                    'expected': expected_annotations[i],
                    'actual': actual_annotations[i]
                })
        
        score = matches / len(expected_annotations) if len(expected_annotations) > 0 else 0
        
        # Print mismatches
        if mismatches:
            print(f"\nFound {len(mismatches)} mismatches:")
            print("-" * 60)
            for mismatch in mismatches:
                print(f"Row {mismatch['row']:2d}: Expected '{mismatch['expected']}', Got '{mismatch['actual']}'")
            print("-" * 60)
        else:
            print("All annotations match!")
        
        return {
            'file_path': csv_path,
            'score': score,
            'matches': matches,
            'total_expected': len(expected_annotations),
            'total_compared': total_compared,
            'accuracy_percentage': score * 100,
            'file_rows': len(df),
            'missing_rows': max(0, len(expected_annotations) - len(actual_annotations)),
            'mismatches': mismatches
        }
        
    except Exception as e:
        return {
            'error': f'Error reading file: {str(e)}',
            'score': 0,
            'total': len(expected_annotations),
            'matches': 0
        }

In [ ]:
score_annotation_file(annotated_manual_annotation_path, test_expected_annotations)


Found 18 mismatches:
------------------------------------------------------------
Row  3: Expected 'recovery', Got 'Correction'
Row  4: Expected 'recovery', Got 'Correction'
Row  6: Expected 'override', Got 'Correction'
Row 10: Expected 'override', Got 'Correction'
Row 15: Expected 'correction', Got 'Recovery'
Row 20: Expected 'follow through', Got 'Correction'
Row 22: Expected 'follow through', Got 'Correction'
Row 25: Expected 'correction', Got 'Override'
Row 27: Expected 'correction', Got 'Recovery'
Row 28: Expected 'correction', Got 'Recovery'
Row 29: Expected 'correction', Got 'Recovery'
Row 40: Expected 'recovery', Got 'Correction'
Row 41: Expected 'recovery', Got 'Correction'
Row 42: Expected 'override', Got 'Correction'
Row 44: Expected 'override', Got 'Follow Through'
Row 46: Expected 'override', Got 'Follow Through'
Row 53: Expected 'correction', Got 'Recovery'
Row 54: Expected 'override', Got 'Correction'
------------------------------------------------------------


{'file_path': '/nas/ucb/daniel_d_kang/arithmetic-reasoning-causality/experiments/token_intervention/output/GPT-OSS_stepwise/annotated_generation_h_filtered_for_manual_annotation.csv',
 'score': 0.7,
 'matches': 42,
 'total_expected': 60,
 'total_compared': 60,
 'accuracy_percentage': 70.0,
 'file_rows': 60,
 'missing_rows': 0,
 'mismatches': [{'row': 3, 'expected': 'recovery', 'actual': 'Correction'},
  {'row': 4, 'expected': 'recovery', 'actual': 'Correction'},
  {'row': 6, 'expected': 'override', 'actual': 'Correction'},
  {'row': 10, 'expected': 'override', 'actual': 'Correction'},
  {'row': 15, 'expected': 'correction', 'actual': 'Recovery'},
  {'row': 20, 'expected': 'follow through', 'actual': 'Correction'},
  {'row': 22, 'expected': 'follow through', 'actual': 'Correction'},
  {'row': 25, 'expected': 'correction', 'actual': 'Override'},
  {'row': 27, 'expected': 'correction', 'actual': 'Recovery'},
  {'row': 28, 'expected': 'correction', 'actual': 'Recovery'},
  {'row': 29, 'exp

In [ ]:
score_annotation_file(eval_annotated_manual_annotation_path, eval_expected_annotations)


Found 8 mismatches:
------------------------------------------------------------
Row  4: Expected 'recovery', Got 'Correction'
Row  6: Expected 'override', Got 'Correction'
Row 10: Expected 'override', Got 'Correction'
Row 16: Expected 'recovery', Got 'Correction'
Row 18: Expected 'override', Got 'Correction'
Row 22: Expected 'override', Got 'Correction'
Row 28: Expected 'recovery', Got 'Correction'
Row 30: Expected 'override', Got 'Correction'
------------------------------------------------------------


{'file_path': '/nas/ucb/daniel_d_kang/arithmetic-reasoning-causality/experiments/token_intervention/output/GPT-OSS_stepwise/annotated_generation_h_filtered_for_manual_annotation.csv',
 'score': 0.8666666666666667,
 'matches': 52,
 'total_expected': 60,
 'total_compared': 60,
 'accuracy_percentage': 86.66666666666667,
 'file_rows': 60,
 'missing_rows': 0,
 'mismatches': [{'row': 4, 'expected': 'recovery', 'actual': 'Correction'},
  {'row': 6, 'expected': 'override', 'actual': 'Correction'},
  {'row': 10, 'expected': 'override', 'actual': 'Correction'},
  {'row': 16, 'expected': 'recovery', 'actual': 'Correction'},
  {'row': 18, 'expected': 'override', 'actual': 'Correction'},
  {'row': 22, 'expected': 'override', 'actual': 'Correction'},
  {'row': 28, 'expected': 'recovery', 'actual': 'Correction'},
  {'row': 30, 'expected': 'override', 'actual': 'Correction'}]}

### Aggregate statistics

In [ ]:
# Read the CSV file
df = pd.read_csv(annotated_generation_path)

# Count annotations
annotation_counts = df['annotation'].value_counts()

print("Annotation counts:")
print("-" * 30)
for annotation, count in annotation_counts.items():
    print(f"{annotation}: {count}")

print(f"\nTotal rows: {len(df)}")

# Calculate percentages
print("\nAnnotation percentages:")
print("-" * 30)
for annotation, count in annotation_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{annotation}: {percentage:.1f}%")

Annotation counts:
------------------------------
Correction: 2046
Recovery: 483
Follow Through: 359
Override: 184

Total rows: 3072

Annotation percentages:
------------------------------
Correction: 66.6%
Recovery: 15.7%
Follow Through: 11.7%
Override: 6.0%
